# Módulo 12 — Virtualización y Despliegue en la Nube
### Parte 1: del modelo entrenado a una app pública con Vercel

**Objetivo:** que el modelo que entrenamos deje de vivir en un notebook y se convierta en una **aplicación web con URL pública**, sin instalar nada en tu computador.

Vamos a desplegar la **regresión lineal del Módulo 6** (valor de vivienda en California), la misma que en el Módulo 10 encapsulamos en la clase `RegresionLineal`.

**El recorrido de hoy:**

```
 Colab                 Archivo            Código              GitHub           Vercel
 ─────                 ───────            ──────              ──────           ──────
 entrenar modelo  ──▶  modelo.json  ──▶  API (FastAPI)  ──▶  repositorio ──▶  URL pública
                                         + página web
```

**Contenido:**
1. Entrenar el modelo (repaso rápido)
2. ¿Qué *es* un modelo entrenado? → exportarlo a `modelo.json`
3. Verificar que el JSON predice igual que el objeto
4. Construir la app: API + página web
5. Probar la app aquí mismo, antes de desplegarla
6. Descargar los archivos
7. Desplegar en Vercel (paso a paso)
8. Retos y preguntas
9. Cierre del curso

> 💡 **En CRISP-DM**, esto es la fase de **Despliegue**: la única que todavía no habíamos cerrado. Un modelo que nadie puede usar no genera valor.

## 0. Preparar el entorno

Además de las librerías de siempre, instalamos **FastAPI**, el framework con el que vamos a convertir el modelo en una API web, y `uvicorn`, el servidor que la ejecuta.

In [ ]:
!pip install -q fastapi uvicorn httpx

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

print("Listo ✔")

## 1. Entrenar el modelo (repaso del Módulo 6 y 10)

Usamos los mismos datos de siempre: **California Housing** (censo de 1990). Cada fila es un sector de California y queremos predecir `MedHouseVal`, el valor medio de la vivienda en **cientos de miles de dólares**.

| Variable | Significado |
|---|---|
| `MedInc` | Ingreso medio del sector (decenas de miles de USD) |
| `HouseAge` | Edad media de las viviendas (años) |
| `AveRooms` | Habitaciones promedio por vivienda |
| `AveBedrms` | Dormitorios promedio por vivienda |
| `Population` | Población del sector |
| `AveOccup` | Personas promedio por hogar |
| `Latitude`, `Longitude` | Ubicación |

In [ ]:
datos = fetch_california_housing(as_frame=True)
X = datos.data
y = datos.target.values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalización: se calcula SOLO con los datos de entrenamiento
media = X_train.mean().values
desviacion = X_train.std().values

Xs_train = (X_train.values - media) / desviacion
Xs_test = (X_test.values - media) / desviacion

print("Entrenamiento:", Xs_train.shape, "| Prueba:", Xs_test.shape)

Traemos la clase del **Módulo 10**, sin cambios. Recuerda: el aprendizaje es descenso de gradiente sobre el MSE, exactamente lo que hicimos a mano en el Módulo 6.

In [ ]:
class RegresionLineal:
    # Regresión lineal entrenada con descenso de gradiente (Módulo 6 → Módulo 10)

    def __init__(self, tasa=0.1, iteraciones=1000):
        self.tasa = tasa
        self.iteraciones = iteraciones
        self.w = None
        self.b = 0.0
        self.historial = []

    def fit(self, X, y):
        n, d = X.shape
        self.w, self.b, self.historial = np.zeros(d), 0.0, []
        for _ in range(self.iteraciones):
            error = (X @ self.w + self.b) - y
            self.w -= self.tasa * (X.T @ error) / n
            self.b -= self.tasa * error.mean()
            self.historial.append(np.mean(error ** 2))
        return self

    def predict(self, X):
        return X @ self.w + self.b

    def score(self, X, y):
        y_hat = self.predict(X)
        return 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)


modelo = RegresionLineal(tasa=0.1, iteraciones=1000).fit(Xs_train, y_train)
r2 = modelo.score(Xs_test, y_test)

plt.figure(figsize=(6, 3))
plt.plot(modelo.historial)
plt.xlabel("Iteración"); plt.ylabel("MSE"); plt.title("Curva de aprendizaje")
plt.tight_layout(); plt.show()

print(f"R² en prueba: {r2:.3f}")

Un R² cercano a 0.6 es lo esperado para una regresión lineal con estos datos. No buscamos el mejor modelo posible: buscamos **llevar un modelo real a producción**.

## 2. ¿Qué *es* un modelo entrenado?

Después de 1000 iteraciones, todo lo que el modelo "aprendió" está guardado en unos pocos números. Mirémoslos:

In [ ]:
pd.DataFrame({
    "variable": X.columns,
    "media (μ)": media.round(3),
    "desviación (σ)": desviacion.round(3),
    "peso (w)": modelo.w.round(4),
}).set_index("variable")

Para predecir una vivienda nueva solo necesitamos:

$$\hat{y} = \sum_{i} w_i \cdot \frac{x_i - \mu_i}{\sigma_i} + b$$

- $\mu_i$ y $\sigma_i$ → cómo se normalizó cada variable
- $w_i$ y $b$ → lo que aprendió el descenso de gradiente

Es la misma ecuación de ERA 1 (álgebra lineal), con parámetros encontrados en ERA 2 (optimización). Así que **el modelo completo cabe en un archivo de texto**. Lo guardamos en formato JSON, que cualquier lenguaje puede leer.

> ⚠️ Si olvidas guardar $\mu$ y $\sigma$, el modelo en producción recibirá datos sin normalizar y dará predicciones absurdas. Es el error más común al desplegar.

In [ ]:
modelo_json = {
    "nombre": "Regresión lineal — valor de vivienda en California",
    "variables": list(X.columns),
    "media": media.tolist(),
    "desviacion": desviacion.tolist(),
    "pesos": modelo.w.tolist(),
    "sesgo": float(modelo.b),
    "r2_prueba": round(float(r2), 4),
    "unidad_salida": "cientos de miles de USD (censo de 1990)",
}

Path("modelo.json").write_text(json.dumps(modelo_json, indent=2, ensure_ascii=False), encoding="utf-8")

print(Path("modelo.json").read_text()[:400], "...")
print(f"\nTamaño del archivo: {Path('modelo.json').stat().st_size} bytes")

Compara ese tamaño con lo que ocuparía llevar las librerías que usamos para entrenar:

In [ ]:
import sklearn, scipy

def tamano_mb(modulo):
    carpeta = Path(modulo.__file__).parent
    return sum(f.stat().st_size for f in carpeta.rglob("*") if f.is_file()) / 1e6

for m in (np, scipy, sklearn):
    print(f"{m.__name__:8s} ≈ {tamano_mb(m):6.0f} MB")

En producción **no necesitamos nada de eso**: la ecuación se calcula con una suma en Python puro. App más liviana = despliegue más rápido, más barato y con menos cosas que se pueden romper.

## 3. Verificar: ¿el JSON predice igual que el objeto?

Escribimos la función que usará la app. Fíjate que **no usa NumPy**: solo lee el JSON y aplica la ecuación.

In [ ]:
def predecir(datos, modelo):
    # ŷ = b + Σ w_i · (x_i − μ_i) / σ_i
    y = modelo["sesgo"]
    for nombre, mu, sigma, w in zip(modelo["variables"], modelo["media"],
                                    modelo["desviacion"], modelo["pesos"]):
        y += w * (datos[nombre] - mu) / sigma
    return y


modelo_cargado = json.loads(Path("modelo.json").read_text())

muestras = X_test.head(5)
comparacion = pd.DataFrame({
    "objeto (NumPy)": modelo.predict((muestras.values - media) / desviacion),
    "JSON (Python puro)": [predecir(fila, modelo_cargado) for fila in muestras.to_dict("records")],
    "valor real": y_test[:5],
}).round(4)

display(comparacion)
assert np.allclose(comparacion.iloc[:, 0], comparacion.iloc[:, 1]), "¡No coinciden! Revisa la normalización"
print("✔ El modelo exportado predice exactamente igual")

## 4. Construir la app

La app tiene dos partes:

| Archivo | Qué hace |
|---|---|
| `main.py` | La **API**: carga `modelo.json` y responde predicciones |
| `index.html` | La **página**: un formulario que le pregunta a la API |
| `modelo.json` | El modelo (lo acabamos de crear) |
| `requirements.txt` | Las librerías que necesita el servidor |

Así funciona una predicción:

```
 Navegador                          Servidor (Vercel)
 ─────────                          ─────────────────
 index.html  ── POST /predict ──▶   main.py
 (formulario)   {"MedInc": 3.5,…}     lee modelo.json
                                      calcula ŷ
             ◀── {"valor_usd": …} ──
```

Usamos `%%writefile`: escribe el contenido de la celda en un archivo, en vez de ejecutarlo.

In [ ]:
carpeta_app = Path("app-vivienda")
carpeta_app.mkdir(exist_ok=True)
shutil.copy("modelo.json", carpeta_app / "modelo.json")
print("Carpeta creada:", carpeta_app.resolve())

### 4.1 La API: `main.py`

Tres "rutas" (direcciones dentro de la app):

- `GET /` → entrega la página web
- `GET /modelo` → información del modelo
- `POST /predict` → recibe los datos de una vivienda y devuelve la predicción

La clase `Vivienda` le dice a FastAPI qué datos esperar. Si alguien envía texto donde va un número, FastAPI lo rechaza automáticamente con un error 422 — validación gratis.

In [ ]:
%%writefile app-vivienda/main.py
"""
Módulo 12 — API del modelo de regresión lineal (California Housing)

Rutas:
    GET  /          → página con el formulario
    GET  /modelo    → información del modelo
    POST /predict   → predicción para una vivienda
    GET  /docs      → documentación automática (la genera FastAPI)
"""
import json
from pathlib import Path

from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

CARPETA = Path(__file__).parent
MODELO = json.loads((CARPETA / "modelo.json").read_text(encoding="utf-8"))

app = FastAPI(
    title="Estimador de valor de vivienda",
    description="Regresión lineal del Módulo 6, desplegada en el Módulo 12.",
    version="1.0",
)


class Vivienda(BaseModel):
    MedInc: float = Field(..., description="Ingreso medio del sector (decenas de miles de USD)", examples=[3.5])
    HouseAge: float = Field(..., description="Edad media de las viviendas (años)", examples=[29])
    AveRooms: float = Field(..., description="Habitaciones promedio por vivienda", examples=[5.2])
    AveBedrms: float = Field(..., description="Dormitorios promedio por vivienda", examples=[1.05])
    Population: float = Field(..., description="Población del sector", examples=[1170])
    AveOccup: float = Field(..., description="Personas promedio por hogar", examples=[2.8])
    Latitude: float = Field(..., description="Latitud", examples=[34.05])
    Longitude: float = Field(..., description="Longitud", examples=[-118.24])


def predecir(datos: dict) -> float:
    # ŷ = b + Σ w_i · (x_i − μ_i) / σ_i   (la misma ecuación del notebook)
    y = MODELO["sesgo"]
    for nombre, mu, sigma, w in zip(MODELO["variables"], MODELO["media"],
                                    MODELO["desviacion"], MODELO["pesos"]):
        y += w * (datos[nombre] - mu) / sigma
    return y


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
def pagina():
    return (CARPETA / "index.html").read_text(encoding="utf-8")


@app.get("/modelo")
def info_modelo():
    return {
        "nombre": MODELO["nombre"],
        "variables": MODELO["variables"],
        "r2_prueba": MODELO["r2_prueba"],
        "unidad_salida": MODELO["unidad_salida"],
    }


@app.post("/predict")
def predict(vivienda: Vivienda):
    y = predecir(vivienda.model_dump())
    y = max(y, 0.0)  # una recta puede dar negativos; un precio no
    return {
        "prediccion": round(y, 4),
        "valor_usd": round(y * 100_000),
        "nota": "Valor estimado con datos del censo de 1990",
    }

### 4.2 La página: `index.html`

HTML con un poco de JavaScript. Lo importante está al final: `fetch("/predict", …)` envía los datos del formulario a la API y muestra la respuesta. Para no pedir latitud y longitud a mano, el usuario elige una ciudad.

In [ ]:
%%writefile app-vivienda/index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Estimador de vivienda</title>
  <style>
    :root { --fondo:#f4f6f8; --tarjeta:#fff; --texto:#1d2733; --suave:#5b6775; --acento:#0f6e8c; --borde:#d8dee5; }
    * { box-sizing: border-box; }
    body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", sans-serif; background:var(--fondo); color:var(--texto); }
    main { max-width: 760px; margin: 0 auto; padding: 32px 16px; }
    h1 { margin: 0 0 4px; font-size: 1.6rem; }
    .sub { color: var(--suave); margin: 0 0 24px; }
    .tarjeta { background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 12px; padding: 24px; }
    .campos { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 16px; }
    label { display: block; font-size: .85rem; font-weight: 600; margin-bottom: 6px; }
    small { display: block; color: var(--suave); font-weight: 400; }
    input, select { width: 100%; padding: 10px; font-size: 1rem; border: 1px solid var(--borde); border-radius: 8px; }
    button { margin-top: 20px; width: 100%; padding: 12px; font-size: 1rem; font-weight: 600;
             color: #fff; background: var(--acento); border: 0; border-radius: 8px; cursor: pointer; }
    button:disabled { opacity: .6; }
    #resultado { margin-top: 20px; padding: 20px; border-radius: 10px; background: #e8f3f6; display: none; }
    #resultado .valor { font-size: 2rem; font-weight: 700; color: var(--acento); }
    footer { margin-top: 20px; font-size: .85rem; color: var(--suave); }
    a { color: var(--acento); }
  </style>
</head>
<body>
<main>
  <h1>🏠 Estimador de valor de vivienda</h1>
  <p class="sub">Regresión lineal entrenada con descenso de gradiente · California, censo de 1990</p>

  <form id="formulario" class="tarjeta">
    <div class="campos">
      <div>
        <label for="ciudad">Ciudad</label>
        <select id="ciudad">
          <option value="34.05,-118.24">Los Ángeles</option>
          <option value="37.77,-122.42">San Francisco</option>
          <option value="32.72,-117.16">San Diego</option>
          <option value="38.58,-121.49">Sacramento</option>
          <option value="36.74,-119.79">Fresno</option>
        </select>
      </div>
      <div>
        <label for="MedInc">Ingreso medio del sector <small>decenas de miles de USD (3.5 = 35.000)</small></label>
        <input id="MedInc" type="number" step="0.1" min="0" value="3.5" required>
      </div>
      <div>
        <label for="HouseAge">Edad de las viviendas <small>años</small></label>
        <input id="HouseAge" type="number" step="1" min="0" value="29" required>
      </div>
      <div>
        <label for="AveRooms">Habitaciones por vivienda <small>promedio</small></label>
        <input id="AveRooms" type="number" step="0.1" min="0" value="5.2" required>
      </div>
      <div>
        <label for="AveBedrms">Dormitorios por vivienda <small>promedio</small></label>
        <input id="AveBedrms" type="number" step="0.05" min="0" value="1.05" required>
      </div>
      <div>
        <label for="Population">Población del sector <small>personas</small></label>
        <input id="Population" type="number" step="10" min="0" value="1170" required>
      </div>
      <div>
        <label for="AveOccup">Personas por hogar <small>promedio</small></label>
        <input id="AveOccup" type="number" step="0.1" min="0" value="2.8" required>
      </div>
    </div>
    <button id="boton" type="submit">Estimar valor</button>

    <div id="resultado">
      <div>Valor estimado de la vivienda:</div>
      <div class="valor" id="valor"></div>
      <div id="detalle"></div>
    </div>
  </form>

  <footer>
    <span id="info-modelo">Cargando información del modelo…</span> ·
    <a href="/docs">Ver la API (/docs)</a>
  </footer>
</main>

<script>
  const campos = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"];

  // Información del modelo (R²) desde la API
  fetch("/modelo").then(r => r.json()).then(m => {
    document.getElementById("info-modelo").textContent = `R² en prueba: ${m.r2_prueba}`;
  });

  document.getElementById("formulario").addEventListener("submit", async (evento) => {
    evento.preventDefault();
    const boton = document.getElementById("boton");
    boton.disabled = true;

    // 1. Reunir los datos del formulario
    const [lat, lon] = document.getElementById("ciudad").value.split(",").map(Number);
    const datos = { Latitude: lat, Longitude: lon };
    for (const c of campos) datos[c] = Number(document.getElementById(c).value);

    // 2. Enviarlos a la API
    try {
      const respuesta = await fetch("/predict", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(datos),
      });
      if (!respuesta.ok) throw new Error(`Error ${respuesta.status}`);
      const r = await respuesta.json();

      // 3. Mostrar el resultado
      document.getElementById("valor").textContent = "$ " + r.valor_usd.toLocaleString("es-CO") + " USD";
      document.getElementById("detalle").textContent = `ŷ = ${r.prediccion} (cientos de miles de USD) · ${r.nota}`;
    } catch (e) {
      document.getElementById("valor").textContent = "No se pudo calcular";
      document.getElementById("detalle").textContent = e.message;
    }
    document.getElementById("resultado").style.display = "block";
    boton.disabled = false;
  });
</script>
</body>
</html>

### 4.3 Dependencias y README

`requirements.txt` le dice a Vercel qué instalar. Fíjate lo corto que es: **ni NumPy, ni scikit-learn**.

In [ ]:
%%writefile app-vivienda/requirements.txt
fastapi
uvicorn

In [ ]:
%%writefile app-vivienda/README.md
# Estimador de valor de vivienda — Módulo 12

Regresión lineal (Módulo 6) entrenada con descenso de gradiente y desplegada como API con FastAPI.

## Enlaces
- App: https://TU-PROYECTO.vercel.app
- Documentación de la API: https://TU-PROYECTO.vercel.app/docs

## Archivos
| Archivo | Rol |
|---|---|
| `modelo.json` | Pesos, sesgo, medias y desviaciones del modelo entrenado |
| `main.py` | API: carga el modelo y responde en `/predict` |
| `index.html` | Página con el formulario |
| `requirements.txt` | Dependencias del servidor |

## Reflexión
(Escribe aquí tus respuestas a las preguntas del final del notebook)

## 5. Probar la app antes de desplegarla

Nunca se despliega algo sin probarlo. FastAPI trae un `TestClient` que simula peticiones web **sin necesidad de levantar un servidor**.

In [ ]:
import sys, importlib
sys.path.insert(0, str(carpeta_app.resolve()))
import main
importlib.reload(main)   # por si modificas main.py y vuelves a ejecutar

from fastapi.testclient import TestClient
cliente = TestClient(main.app)

# 1. ¿Carga la página?
print("GET /        →", cliente.get("/").status_code)

# 2. ¿Responde la información del modelo?
print("GET /modelo  →", cliente.get("/modelo").json())

# 3. Una predicción: vivienda típica en Los Ángeles
vivienda = {"MedInc": 3.5, "HouseAge": 29, "AveRooms": 5.2, "AveBedrms": 1.05,
            "Population": 1170, "AveOccup": 2.8, "Latitude": 34.05, "Longitude": -118.24}
print("POST /predict →", cliente.post("/predict", json=vivienda).json())

¿Y si alguien envía un dato mal? Probemos con texto en lugar de número:

In [ ]:
malo = dict(vivienda, MedInc="mucho")
r = cliente.post("/predict", json=malo)
print("Código:", r.status_code)
print(r.json()["detail"][0]["msg"])

FastAPI rechazó la petición con un **422** antes de llegar al modelo. Esa validación sale "gratis" de la clase `Vivienda`.

### (Opcional) Ver la página dentro de Colab

Esta celda levanta el servidor real en segundo plano y abre la app en una ventana. Solo funciona en Google Colab.

In [ ]:
import subprocess, time

servidor = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"], cwd=carpeta_app)
time.sleep(3)

try:
    from google.colab import output
    output.serve_kernel_port_as_window(8000)
except ImportError:
    print("No estás en Colab: abre http://localhost:8000 en tu navegador")

## 6. Descargar los archivos

Empaquetamos la carpeta en un `.zip`. Descomprímelo en tu computador: verás los 5 archivos que subirás a GitHub.

In [ ]:
# __pycache__ la crea Python al importar main.py en las pruebas: no se sube a GitHub
shutil.rmtree(carpeta_app / "__pycache__", ignore_errors=True)

shutil.make_archive("app-vivienda", "zip", carpeta_app)
import zipfile
print("Contenido del zip:", sorted(zipfile.ZipFile(f"{carpeta_app}.zip").namelist()))

try:
    from google.colab import files
    files.download("app-vivienda.zip")
except ImportError:
    print("Archivo listo: app-vivienda.zip")

## 7. Desplegar en Vercel

No necesitas instalar nada: todo se hace desde el navegador.

### Paso 1 — Subir los archivos a GitHub

1. Entra a [github.com](https://github.com) e inicia sesión (o crea una cuenta gratis).
2. Botón **New** (repositorio nuevo). Nombre: `app-vivienda`. Visibilidad: **Public**. Crear.
3. En el repositorio vacío, haz clic en el enlace **uploading an existing file**.
4. Arrastra los **5 archivos** (no la carpeta) y confirma con **Commit changes**.

> ⚠️ Los archivos deben quedar en la **raíz** del repositorio. Si quedan dentro de una subcarpeta, Vercel no encuentra `main.py`.

### Paso 2 — Conectar Vercel

1. Entra a [vercel.com](https://vercel.com) y regístrate **con tu cuenta de GitHub** (plan Hobby, gratuito).
2. **Add New… → Project**.
3. Busca `app-vivienda` en la lista de repositorios y presiona **Import**.
4. Vercel detecta que es una app **FastAPI**: encuentra `main.py` y la variable `app` dentro de él. No hay que configurar nada.
5. Presiona **Deploy** y espera un minuto.

### Paso 3 — Probar

Vercel te entrega una URL del tipo `https://app-vivienda-xxxx.vercel.app`. Revisa:

| Dirección | Debe mostrar |
|---|---|
| `/` | El formulario |
| `/docs` | La documentación interactiva de la API |
| `/modelo` | El JSON con el R² |

Ábrela desde tu celular. Compártela con un compañero. **Tu modelo ya está en producción.**

### Paso 4 — Despliegue continuo

En GitHub, abre `index.html`, edítalo (el lápiz ✏️), cambia el título y haz commit. Vuelve a Vercel: **se redesplegó solo**. Cada cambio que subas a GitHub actualiza la app automáticamente.

### Si algo falla

| Síntoma | Causa probable |
|---|---|
| Vercel no detecta FastAPI | Los archivos están en una subcarpeta, no en la raíz |
| Error 500 al abrir la página | Falta `modelo.json` o `index.html` en el repo. Revisa la pestaña **Logs** del proyecto |
| Error 422 en `/predict` | Los nombres de las variables no coinciden con la clase `Vivienda` |
| Predicciones absurdas | El JSON no tiene las medias/desviaciones correctas |

## 8. Retos y preguntas

Responde en el `README.md` de tu repositorio:

1. **¿Por qué no pusimos `scikit-learn` en `requirements.txt`?** ¿Qué pasaría con el tamaño de la app y el tiempo de despliegue si lo hiciéramos?
2. **Reentrena y redespliega.** Cambia `iteraciones=1000` por `iteraciones=20`, vuelve a exportar `modelo.json`, súbelo a GitHub y compara la predicción para la misma vivienda. ¿Qué te dice eso de la curva de aprendizaje?
3. **Rompe la validación.** En `/docs`, usa *Try it out* en `/predict` y envía un número negativo de habitaciones. ¿El modelo lo acepta? ¿Debería? ¿Cómo lo impedirías en la clase `Vivienda`? (Pista: `Field(..., ge=0)`)
4. **Piensa en otro modelo.** Una regresión cabe en un JSON de ~1 KB. ¿Qué tendrías que guardar para desplegar el **KNN** del Módulo 8? ¿Por qué ese modelo pesa mucho más?

## 9. Cierre del curso

Lo que acabas de desplegar es el recorrido completo del curso en un solo archivo:

| ERA | Qué aportó | Dónde está en tu app |
|---|---|---|
| **ERA 1** — Matemática | Álgebra lineal, derivadas, gradiente | La ecuación $\hat{y} = \sum w_i x_i + b$ dentro de `predecir()` |
| **ERA 2** — Modelos | Descenso de gradiente, regresión, métricas | Los pesos y el R² dentro de `modelo.json` |
| **ERA 3** — Arquitectura | Clases (M10), rendimiento (M11), despliegue (M12) | `RegresionLineal`, una app liviana y una URL pública |

**Próxima sesión:** empaquetaremos esta misma app en un **contenedor Docker** para poder ejecutarla en cualquier servidor, propio o en la nube, y compararemos con lo que Vercel hizo hoy por nosotros sin que lo viéramos.